In [ ]:
# ============================================================
# SETUP - Correr esta celda primero
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install torch_geometric

In [ ]:
import os
import numpy as np
import pandas as pd
import copy
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import precision_recall_curve
from torch_geometric.loader import DataLoader


In [ ]:
from utils.datasets   import NF_IDS_Dataset
from utils.models     import (
    StaticGNN, StaticGNN_biasinit,
    ST_GNN, ST_GNN_biasinit, ST_GNN_Robust,
)
from utils.metrics    import calculate_metrics_gnn
from utils.training   import train_epoch, evaluate, find_optimal_threshold
from utils.experiment import ExperimentManager


# Functions

This handles the complex logic: Truncated Backpropagation for ST-GNN and detailed metric calculations (including False Positives).

## pos_weight

In [ ]:
def check_class_balance(loader, name="Loader"):
    total_benign = 0
    total_attack = 0

    print(f"--- Analyzing {name} ---")
    for data in loader:
        # data.y is [1] or [N]
        y = data.y.view(-1)
        n_ones = y.sum().item()
        n_zeros = y.shape[0] - n_ones

        total_attack += n_ones
        total_benign += n_zeros

    print(f"Benign: {int(total_benign)}")
    print(f"Attack : {int(total_attack)}")

    if total_attack > 0:
        # POS_WEIGHT: (Negative / Positive)
        # Used to penalize errors in the minority class more severely
        ratio = total_benign / total_attack
        pos_weight_tensor = torch.tensor([ratio])

        # BIAS_INIT: log(Positives / Negatives)
        # Used to allow the network to "guess" the base probability from the start
        # Mathematically equivalent to: -math.log(ratio)
        bias_val = math.log(total_attack / total_benign)

        print(f"Ratio (pos_weight): {ratio:.2f}")
        print(f"Bias Init suggested: {bias_val:.4f}")

        return pos_weight_tensor, bias_val
    else:
        print("ALERT: There are no attacks on this dataset.")
        return None, None



# Auxiliary

In [10]:
ROOT_PATH = "./dataset_processed"

In [11]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


In [ ]:
pos_weight_value, bias_value = check_class_balance(train_loader, "TRAIN")
print(f"pos_weight_value:{pos_weight_value}\n")
print(f"bias_value:{bias_value}\n")

check_class_balance(val_loader, "VAL")

--- Analyzing TRAIN ---
Benign: 992229
Attack : 49564
Ratio (pos_weight): 20.02
Bias Init suggested: -2.9967
pos_weight_value:tensor([20.0191])

bias_value:-2.9966891636638033

--- Analyzing VAL ---
Benign: 757206
Attack : 31329
Ratio (pos_weight): 24.17
Bias Init suggested: -3.1851


(tensor([24.1695]), -3.1850911570685216)

# Main

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.001

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
OUT_DIM = 1     # Binary output (logit): >0 is attack, <0 is benign
DROPOUT = 0.2

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


Using device: cpu


## Train static

### Bias init

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 30
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5




Using device: cpu


In [ ]:
model_config = {
    "model_name": "StaticGNN_biasinit",
    "type": "GNN_traditional_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit.csv")

In [ ]:
static_bias_model = StaticGNN_biasinit(**model_config['model_params']).to(DEVICE)

weight = torch.tensor([POS_WEIGHT]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

opt_static = torch.optim.Adam(static_bias_model.parameters(), lr=LR)


best_aucpr_static_bias = 0.0
best_wts_static_bias = copy.deepcopy(static_bias_model.state_dict())
best_metrics_static_bias = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(static_bias_model, train_loader, opt_static, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
    metrics_static_bias = evaluate(static_bias_model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics_static_bias['AUC-PR']:.4f} | Val F1: {metrics_static_bias['F1']:.4f} | Val Rec: {metrics_static_bias['Recall']:.4f}")

    if metrics_static_bias['AUC-PR'] > best_aucpr_static_bias:
        best_aucpr_static_bias = metrics_static_bias['AUC-PR']
        best_metrics_static_bias = metrics_static_bias
        best_wts_static_bias = copy.deepcopy(static_bias_model.state_dict())
        print(f"New record AUC-PR: {best_aucpr_static_bias:.4f} (FPR: {metrics_static_bias['FPR']:.4f})")


print(f"\nRestoring better weights (AUC-PR: {best_aucpr_static_bias:.4f})...")
static_bias_model.load_state_dict(best_wts_static_bias)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics_static_bias, model_object=static_bias_model)

print(f"\nBest AUC-PR for Static GNN biasinit obtained: {best_aucpr_static_bias:.4f}")

In [ ]:
optimal_thresh = find_optimal_threshold(static_bias_model, val_loader, DEVICE)

## Train ST-GNN

### Bias init

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32, 64]
}


Using device: cpu


In [ ]:
model_config = {
    "model_name": "ST_GNN_biasinit",
    "type": "GNN_and_GRU_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit.csv")

In [ ]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"ST_GNN_H{h_dim}_BiasOn"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f"[{count}/2] Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n[{count}/2] Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = ST_GNN_biasinit(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, True)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()


History loaded. 2 experiments already performed

ST_GNN_H32_BiasOn

[1/2] Starting: ST_GNN_H32_BiasOn
   Ep 2 | Loss: 0.2209 | Val AUC-PR: 0.0335 | Val Rec: 0.0000
   Ep 4 | Loss: 0.2192 | Val AUC-PR: 0.0541 | Val Rec: 0.0000
   Ep 6 | Loss: 0.2161 | Val AUC-PR: 0.0568 | Val Rec: 0.0000
   Ep 8 | Loss: 0.2149 | Val AUC-PR: 0.0701 | Val Rec: 0.0000
   Ep 10 | Loss: 0.2001 | Val AUC-PR: 0.0646 | Val Rec: 0.0000
   Ep 12 | Loss: 0.2170 | Val AUC-PR: 0.0397 | Val Rec: 0.0000
   Ep 14 | Loss: 0.2136 | Val AUC-PR: 0.0797 | Val Rec: 0.0000
   Ep 16 | Loss: 0.2040 | Val AUC-PR: 0.0808 | Val Rec: 0.0000
   Ep 18 | Loss: 0.2135 | Val AUC-PR: 0.1925 | Val Rec: 0.0000
   Ep 20 | Loss: 0.2048 | Val AUC-PR: 0.0760 | Val Rec: 0.0000
   Ep 22 | Loss: 0.1985 | Val AUC-PR: 0.1515 | Val Rec: 0.0000
   Ep 24 | Loss: 0.1951 | Val AUC-PR: 0.2109 | Val Rec: 0.1290
   Ep 26 | Loss: 0.1912 | Val AUC-PR: 0.2063 | Val Rec: 0.1458
   Ep 28 | Loss: 0.1930 | Val AUC-PR: 0.2091 | Val Rec: 0.1464
   Ep 30 | Loss: 0.1

### Bias init - Robust

In [15]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32]#, 64]
}


Using device: cpu


In [16]:
model_config = {
    "model_name": "ST_GNN_biasinit",
    "type": "GNN_and_GRU_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [17]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit_robust.csv")

In [20]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"ST_GNN_H{h_dim}_BiasOn"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f"[{count}/2] Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n[{count}/2] Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = ST_GNN_Robust(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, True)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()


# Experiment recorded in ./logs/experiments_log_gnn_biasinit.csv
# Saved model: ST_GNN_H32_BiasOn_20260128_2215_AUC-PR_0.2414
#    Best AUC-PR: 0.2414 (FPR: 0.0052)

# BEST THRESHOLD FOUND: 0.2916
#    F1 Score : 0.2646
#    Recall   : 0.1739
#    Precision: 0.5538

# Probability Diagnosis:
#    Average assigned to Benignos: 0.0666
#    Average assigned to Attacks : 0.2291


ST_GNN_H32_BiasOn

[1/2] Starting: ST_GNN_H32_BiasOn
   Ep 2 | Loss: 0.2035 | Val AUC-PR: 0.0619 | Val Rec: 0.0000
   Ep 4 | Loss: 0.1893 | Val AUC-PR: 0.0698 | Val Rec: 0.0000
   Ep 6 | Loss: 0.2023 | Val AUC-PR: 0.0601 | Val Rec: 0.0000
   Ep 8 | Loss: 0.2027 | Val AUC-PR: 0.0712 | Val Rec: 0.0000
   Ep 10 | Loss: 0.1938 | Val AUC-PR: 0.0927 | Val Rec: 0.0000
   Ep 12 | Loss: 0.1818 | Val AUC-PR: 0.0803 | Val Rec: 0.0112
   Ep 14 | Loss: 0.1889 | Val AUC-PR: 0.1726 | Val Rec: 0.0773
   Ep 16 | Loss: 0.1864 | Val AUC-PR: 0.2052 | Val Rec: 0.1559
   Ep 18 | Loss: 0.1786 | Val AUC-PR: 0.1852 | Val Rec: 0.1532
   Ep 20 | Loss: 0.1745 | Val AUC-PR: 0.1817 | Val Rec: 0.1528
   Ep 22 | Loss: 0.1750 | Val AUC-PR: 0.2084 | Val Rec: 0.1517
   Ep 24 | Loss: 0.1767 | Val AUC-PR: 0.2050 | Val Rec: 0.1497
   Ep 26 | Loss: 0.1521 | Val AUC-PR: 0.1745 | Val Rec: 0.0943
   Ep 28 | Loss: 0.1638 | Val AUC-PR: 0.1702 | Val Rec: 0.0228
   Ep 30 | Loss: 0.1761 | Val AUC-PR: 0.2004 | Val Rec: 0.1523
   Ep